In [42]:
import random
import time
import math
import numpy as np

import deap
from deap import base, creator, tools

In [43]:
plantType = ["Null", "Corn", "Beans", "Squash"]

DEFAULT_NITROGEN = 50 # 50 ppm as baseline for the soil.
DEFAULT_LIGHT = 1 # 100% sunlight as baseline for the light level.
DEFAULT_YIELD = 0

CELLDISTANCE = 0

def create_plant(plantTypeVal):
    plantInd = [plantType[plantTypeVal], int(DEFAULT_NITROGEN), float(DEFAULT_LIGHT), float(DEFAULT_YIELD)]
    return plantInd

# Modifier Layout: [Plant-Corn Nitrogen, Plant-Corn Light, Plant-Beans Nitrogen, Plant-Beans Light, Plant-Squash Nitrogen, Plant-Squash Light]
CORN_MODIFIER = [-15, -.2, -10, .1, -5, -.25]
BEANS_MODIFIER = [20, 0, -5, -.1, 15, -.05]
SQUASH_MODIFIER = [5, 0, 5, -.05, -8, -.15]

def apply_plant_modifier(castingPlant, targetPlant):
    if (castingPlant[0] == "Null"):
        return targetPlant

    modifier = []
    if (castingPlant[0] == "Corn"):
        modifier = CORN_MODIFIER
    elif (castingPlant[0] == "Beans"):
        modifier = BEANS_MODIFIER
    elif (castingPlant[0] == "Squash"):
        modifier = SQUASH_MODIFIER

    if (targetPlant[0] == "Corn"):
        targetPlant[1] += modifier[0]
        targetPlant[2] += modifier[1]
    elif (targetPlant[0] == "Beans"):
        targetPlant[1] += modifier[2]
        targetPlant[2] += modifier[3]
    elif (targetPlant[0] == "Squash"):
        targetPlant[1] += modifier[4]
        targetPlant[2] += modifier[5]

    # Clamp nitrogen >= 0 and light to [0, 1]
    targetPlant[1] = max(0, targetPlant[1])
    targetPlant[2] = max(0.0, min(1.0, targetPlant[2]))

    return targetPlant

def calculate_all_modifiers(plantArray):
    for row in range(len(plantArray)):
        for col in range(len(plantArray[row])):
            curPlant = plantArray[row][col]
            if (row != 0):
                apply_plant_modifier(plantArray[row-1][col], curPlant)
                if (col != 0):
                    apply_plant_modifier(plantArray[row-1][col-1], curPlant)
                if (col != plantArray.shape[1]-1):
                    apply_plant_modifier(plantArray[row-1][col+1], curPlant)
            if (row != plantArray.shape[0]-1):
                apply_plant_modifier(plantArray[row+1][col], curPlant)
                if (col != 0):
                    apply_plant_modifier(plantArray[row+1][col-1], curPlant)
                if (col != plantArray.shape[1]-1):
                    apply_plant_modifier(plantArray[row+1][col+1], curPlant)
            if (col != 0):
                apply_plant_modifier(plantArray[row][col-1], curPlant)
            if (col != plantArray.shape[1]-1):
                apply_plant_modifier(plantArray[row][col+1], curPlant)

def calculate_yield_val(value, optMin, optMax):
    mid = (optMin + optMax) / 2
    half_range = (optMax - optMin) / 2
    yieldVal = max(0, half_range - abs(value - mid))
    return yieldVal

CORN_NITROGEN_RANGE = [120, 200]
CORN_LIGHT_RANGE = [.8, 1]
BEANS_NITROGEN_RANGE = [30, 60]
BEANS_LIGHT_RANGE = [.5, .75]
SQUASH_NITROGEN_RANGE = [60, 120]
SQUASH_LIGHT_RANGE = [.65, .9]

def calculate_plant_yield(plantInd):
    if (plantInd[0] == "Null"):
        return plantInd

    nitrogenRange = []
    lightRange = []

    if (plantInd[0] == "Corn"):
        nitrogenRange = CORN_NITROGEN_RANGE
        lightRange = CORN_LIGHT_RANGE
    elif (plantInd[0] == "Beans"):
        nitrogenRange = BEANS_NITROGEN_RANGE
        lightRange = BEANS_LIGHT_RANGE
    elif (plantInd[0] == "Squash"):
        nitrogenRange = SQUASH_NITROGEN_RANGE
        lightRange = SQUASH_LIGHT_RANGE

    nitrogenValue = calculate_yield_val(plantInd[1], nitrogenRange[0], nitrogenRange[1])
    lightValue = calculate_yield_val(plantInd[2], lightRange[0], lightRange[1])

    plantInd[3] = nitrogenValue + lightValue
    return plantInd

def calculate_all_yields(plantArray):
    totalYield = 0
    for row in range(len(plantArray)):
        for col in range(len(plantArray[row])):
            calculate_plant_yield(plantArray[row][col])
            totalYield += plantArray[row][col][3]
    return totalYield

In [44]:
# Test functions

def print_plant_array(plantArray):
    cols = len(plantArray[0])
    sep = "+" + ("-" * 16 + "+") * cols
    for row in plantArray:
        print(sep)
        cells = [f" {p[0]:<6} Y:{p[3]:>5.2f} " for p in row]
        print("|" + "|".join(cells) + "|")
    print(sep)

def print_full_plant_array(plantArray):
    cols = len(plantArray[0])
    sep = "+" + ("-" * 32 + "+") * cols
    for row in plantArray:
        print(sep)
        cells = [f" {p[0]:<6} N:{p[1]:>4}  L:{p[2]:.2f}  Y:{p[3]:>5.2f} " for p in row]
        print("|" + "|".join(cells) + "|")
    print(sep)



def test_yields(xSize, ySize):
    fieldArray = np.array([[create_plant(random.randint(0, 3)) for _ in range(ySize)] for _ in range(xSize)], dtype=object)
    calculate_all_modifiers(fieldArray)
    totalYield = calculate_all_yields(fieldArray)
    print_full_plant_array(fieldArray)
    print(f"Total Yield: {totalYield}")

test_yields(5, 5)
    

+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Squash N:  55  L:0.45  Y: 0.00 | Corn   N:  80  L:0.80  Y: 0.00 | Beans  N:  40  L:0.95  Y:10.00 | Squash N:  44  L:0.40  Y: 0.00 | Corn   N:  65  L:1.00  Y: 0.00 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Beans  N:  30  L:1.00  Y: 0.00 | Corn   N:  70  L:0.60  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 | Squash N:  44  L:0.40  Y: 0.00 | Squash N:  29  L:0.45  Y: 0.00 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Corn   N:  45  L:0.60  Y: 0.00 | Squash N:  45  L:0.00  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 

In [45]:
GRID_X = 5
GRID_Y = 5
N_GENES = GRID_X * GRID_Y

POP_SIZE = 100

def evaluate(individual):
    plantArray = np.array([[create_plant(individual[row * GRID_Y + col]) for col in range(GRID_Y)] for row in range(GRID_X)], dtype=object)
    calculate_all_modifiers(plantArray)
    totalYield = calculate_all_yields(plantArray)
    return (totalYield,)

In [46]:
def run_random():
    bestArray = []
    bestYield = -1
    for i in range(POP_SIZE):
        fieldArray = np.array([[create_plant(random.randint(0, 3)) for _ in range(GRID_Y)] for _ in range(GRID_X)], dtype=object)
        calculate_all_modifiers(fieldArray)
        totalYield = calculate_all_yields(fieldArray)
        if (totalYield > bestYield):
            bestYield = totalYield
            bestArray = fieldArray
    print_full_plant_array(bestArray)
    print(f"Best Yield: {bestYield}")

run_random()

+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Squash N:  95  L:0.85  Y:25.05 | Beans  N:  40  L:0.65  Y:10.10 | Beans  N:  40  L:0.65  Y:10.10 | Squash N:  87  L:0.70  Y:27.05 | Squash N:  72  L:0.75  Y:12.10 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Beans  N:  50  L:0.70  Y:10.05 | Beans  N:  50  L:0.55  Y:10.05 | Null   N:  50  L:1.00  Y: 0.00 | Beans  N:  50  L:0.70  Y:10.05 | Beans  N:  50  L:0.85  Y:10.00 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Squash N:  75  L:0.65  Y:15.00 | Null   N:  50  L:1.00  Y: 0.00 | Squash N:  90  L:0.60  Y:30.00 | Corn   N: 125  L:1.00  Y: 5.00 | Squash N:  82  L:0.45  Y:22.00 

In [47]:
def run_greedy():
    bestArray = []
    bestYield = -1
    for i in range(POP_SIZE):
        fieldArray = [0] * N_GENES
        for j in range(N_GENES):
            bestGreedyPlant = -1
            bestPlantYield = -1
            for plantType in range(4):
                fieldArray[j] = plantType
                curPlantYield = evaluate(fieldArray)
                if (curPlantYield[0] > bestPlantYield):
                    bestPlantYield = curPlantYield[0]
                    bestGreedyPlant = plantType
            fieldArray[j] = bestGreedyPlant

        individual = np.array([[create_plant(fieldArray[row * GRID_Y + col]) for col in range (GRID_Y)] for row in range (GRID_X)], dtype=object)
        calculate_all_modifiers(individual)
        totalYield = calculate_all_yields(individual)
        if (totalYield > bestYield):
            bestYield = totalYield
            bestArray = individual
    print_full_plant_array(bestArray)
    print(f"Best Yield: {bestYield}")
run_greedy()

+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Beans  N:  55  L:0.80  Y: 5.00 | Beans  N:  55  L:0.65  Y: 5.10 | Beans  N:  55  L:0.65  Y: 5.10 | Beans  N:  45  L:0.60  Y:15.10 | Beans  N:  45  L:0.75  Y:15.00 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Squash N:  82  L:0.45  Y:22.00 | Squash N:  89  L:0.25  Y:29.00 | Squash N:  89  L:0.25  Y:29.00 | Squash N: 109  L:0.45  Y:11.00 | Beans  N:  45  L:0.60  Y:15.10 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Beans  N:  50  L:0.85  Y:10.00 | Corn   N: 130  L:1.00  Y:10.00 | Null   N:  50  L:1.00  Y: 0.00 | Beans  N:  45  L:0.45  Y:15.00 | Squash N: 102  L:0.65  Y:18.00 

In [48]:
MAX_PASSES = 50

def run_hillclimb():
    max_passes = MAX_PASSES
    moreImprovements = True
    fieldArray = np.random.randint(0, 4, size=(N_GENES))
    passes = 0
    while moreImprovements:
        moreImprovements = False
        for j in range(N_GENES):
            bestGreedyPlant = -1
            bestPlantYield = -1
            for plantType in range(4):
                fieldArray[j] = plantType
                print(fieldArray)
                curPlantYield = evaluate(fieldArray)
                if (curPlantYield[0] > bestPlantYield):
                    bestPlantYield = curPlantYield[0]
                    bestGreedyPlant = plantType
                    moreImprovements = True
            fieldArray[j] = bestGreedyPlant
        passes = passes + 1
        if (passes >= max_passes):
            moreImprovements = False

    individual = np.array([[create_plant(fieldArray[row * GRID_Y + col]) for col in range (GRID_Y)] for row in range (GRID_X)], dtype=object)
    calculate_all_modifiers(individual)
    totalYield = calculate_all_yields(individual)
    print_full_plant_array(individual)
    print(f"Best Yield: {totalYield}")
run_hillclimb()

[0 3 1 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[1 3 1 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 3 1 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[3 3 1 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 0 1 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 1 1 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 1 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 3 1 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 0 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 1 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 2 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 3 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 2 0 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 2 1 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 2 2 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 2 3 3 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 2 3 0 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 2 3 1 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 2 3 2 1 2 3 0 3 2 1 3 2 0 2 3 1 0 3 2 2 1 3 1]
[2 2 2 3 3 1

In [49]:
def run_ga():
    if not hasattr(creator, "FitnessMax"):
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    if not hasattr(creator, "Individual"):
        creator.create("Individual", list, fitness=creator.FitnessMax)

    toolbox = base.Toolbox()

    toolbox.register("attr_plant", random.randint, 0, 3)
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_plant, n=N_GENES)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutUniformInt, low=0, up=3, indpb=0.1)
    toolbox.register("select", tools.selTournament, tournsize=3)
    
    POP_SIZE = 100
    N_GEN = 100
    CROSSOVER = 0.7
    MUTATION = 0.2

    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)

    hof = tools.HallOfFame(1)
    hof.update(pop)

    for gen in range(1, N_GEN + 1):
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))

        # Crossover
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CROSSOVER:
                toolbox.mate(c1, c2)
                del c1.fitness.values
                del c2.fitness.values

        # Mutation
        for mutant in offspring:
            if random.random() < MUTATION:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        invalid = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid:
            ind.fitness.values = toolbox.evaluate(ind)

        pop[:] = offspring

        hof.update(pop)

        fitness = [ind.fitness.values[0] for ind in pop]
        print(f"{gen:>4} {np.mean(fitness):>10.4f} {np.max(fitness):>10.4f}")

    print(f"\nBest Fitness: {hof[0].fitness.values[0]:.4f}")
    best = hof[0]
    bestArray = np.array([[create_plant(best[row * GRID_Y + col]) for col in range(GRID_Y)] for row in range(GRID_X)], dtype=object)
    calculate_all_modifiers(bestArray)
    bestYield = calculate_all_yields(bestArray)
    print_full_plant_array(bestArray)
    print(f"Best Yield: {bestYield}")

    return pop, hof

pop, hof = run_ga()

# continue from here

   1   100.1795   213.2500
   2   123.9495   216.2500
   3   144.9180   238.6000
   4   159.3840   291.4500
   5   184.5945   261.5000
   6   201.3940   371.4000
   7   207.3385   285.5500
   8   224.9320   313.0500
   9   230.1570   346.4500
  10   249.9010   346.4500
  11   253.1270   344.2500
  12   259.8410   340.7500
  13   277.5575   322.8000
  14   280.4295   340.5500
  15   282.4155   341.0000
  16   292.4335   341.6000
  17   304.7365   341.6000
  18   310.2500   346.9500
  19   319.7375   358.7000
  20   320.1715   377.0500
  21   326.5000   377.0500
  22   320.1380   378.9500
  23   335.1815   380.9000
  24   338.3905   382.7000
  25   341.2715   392.8000
  26   345.5250   400.8000
  27   357.4455   400.8000
  28   374.1850   422.8000
  29   377.0545   422.8000
  30   379.2430   422.8000
  31   387.8560   422.8000
  32   392.9725   422.8000
  33   409.9035   422.8000
  34   411.7655   422.8000
  35   404.3645   422.8000
  36   403.6485   422.8000
  37   404.7795   422.8000
 